In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from tqdm import tqdm
from urllib.parse import urlparse
import numpy as np


# DRIVE_BASE = Path('/content/drive/Othercomputers/My MacBook Pro/gdrive-sync')

In [ ]:
PROJECT_CODE = "enguled-bbox-levels-crops-v4-longer-epochs"
TILE_SIZE = 100
BATCH_SIZE = 8

DS = Path("../../datasets")
PROJECT_DIR = DS / "garbage" / "experiments" / PROJECT_CODE
LOG_BASE = PROJECT_DIR / "log"
PROJECT_DIR.mkdir(exist_ok=True, parents=True)
LOG_BASE.mkdir(exist_ok=True, parents=True)


# TACO_DIR = Path("/content/data")
# ANN_FILE = TACO_DIR / "annotations.json"
DATA_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/engulfed-bbox-levels-crops")

In [ ]:
from fastai.vision.all import *
from mtrain.smallnet.unet.train import get_dls, get_learner

# dls = get_dls(BATCH_SIZE, LOG_BASE, TILE_SIZE, DATA_DIR / "images", DATA_DIR / "masks" )

dls = SegmentationDataLoaders.from_label_func(
    LOG_BASE,
    bs=BATCH_SIZE,
    fnames=get_image_files(DATA_DIR / "images"),
    label_func=lambda o: DATA_DIR / "masks" / f"{o.stem}.png",
    codes=np.array(["background", "trash"]),
    item_tfms=Resize(TILE_SIZE),
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    batch_tfms=aug_transforms(),
)
dls.train.prefetch_factor = 4
dls.valid.prefetch_factor = 4

In [ ]:
class CombinedLoss:
    "Dice and Focal combined"
    def __init__(self, axis=1, smooth=1., alpha=1.):
        store_attr()
        self.focal_loss = FocalLossFlat(axis=axis)
        self.dice_loss =  DiceLoss(axis, smooth)

    def __call__(self, pred, targ):
        return self.focal_loss(pred, targ) + self.alpha * self.dice_loss(pred, targ)

    def decodes(self, x):    return x.argmax(dim=self.axis)
    def activation(self, x): return F.softmax(x, dim=self.axis)


In [ ]:
learner = unet_learner(dls, mobilenet_v3_large, loss_func=CombinedLoss())
learner = learner.remove_cb(ProgressCallback)

In [ ]:
learner.fine_tune(5)